In [28]:
import os
import numpy as np
import importlib
import pandas as pd
import obj_2_pcd
importlib.reload(obj_2_pcd)
from SlicedVolumeDataset import SlicedVolumeDataset

tiff_dir_root = '/data/jhahn/data/brain_lightsheet/slices'
obj_dir_root = '/data/jhahn/data/shape_dataset/data/brain_lightsheet_2'

dataset_annotation_file_name = obj_dir_root+"/data.csv"

_df  = obj_2_pcd.create_dataset(dataset_annotation_file_name, tiff_dir_root, obj_dir_root)
#print(_df.head())
data_loader = obj_2_pcd.create_data_loader(dataset_annotation_file_name)
print(f"✅ DataLoader 생성 완료. 총 배치 개수: {len(data_loader)}")

data length: 224
✅ DataLoader 생성 완료. 총 배치 개수: 224


In [29]:
_df

,obj_dir_root,image_filename_list_for_one_sub_brain,slice_angle,tickness,from_index,to_index,num_of_missing_slices_dist,_num_of_missing_slices_list,is_curvature,num_of_slices
0,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.002,0,363,Uniform,"31,31,31,31,31,31,31,31,31,31,31",False,11
1,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.002,0,363,Uniform,"31,31,31,31,31,31,31,31,31,31,31",True,11
2,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.002,0,358,Gaussian,"29,31,32,32,31,29,30,29,31,30,32",False,11
3,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.002,0,358,Gaussian,"29,31,32,32,31,29,30,29,31,30,32",True,11
4,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_0_0,0.002,0,360,Uniform,"28,28,28,28,28,28,28,28,28,28,28,28",False,12
...,...,...,...,...,...,...,...,...,...,...
219,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_1_1,0.002,0,630,Gaussian,"35,35,35,35,35,35,36,34,35,34,35,35,35,36,35,3...",True,17
220,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_1_1,0.002,0,630,Uniform,"33,33,33,33,33,33,33,33,33,33,33,33,33,33,33,3...",False,18
221,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_1_1,0.002,0,630,Uniform,"33,33,33,33,33,33,33,33,33,33,33,33,33,33,33,3...",True,18
222,/data/jhahn/data/shape_dataset/data/brain_ligh...,/data/jhahn/data/brain_lightsheet/slices/slice...,sliced_on_1_1_1,0.002,0,627,Gaussian,"33,33,32,31,33,32,32,33,33,34,34,33,32,32,33,3...",False,18


In [30]:
import torch
import pandas as pd
import numpy as np
import multiprocessing
from tqdm import tqdm
import importlib
import obj_2_pcd
importlib.reload(obj_2_pcd)
# 데이터 변환 (예시)


if torch.cuda.is_available():
    device = torch.device("cuda:0")
    torch.cuda.set_device(device)
else:
    device = torch.device("cpu")


### 2. DataLoader 순회 (Iteration)
tasks_to_run = []
# 일반적으로 훈련 루프(Training Loop)에서 사용됩니다.


# DataLoader를 순회하며 배치 단위로 데이터(이미지)와 레이블을 가져옵니다.
for batch_idx, (tiff_images, labels, output_dir) in enumerate(data_loader):
    
    missing_slices_list = [t.item() for t in labels['missing_slices_list']]
    _tiff_images = [t[0] for t in tiff_images]
    _output_dir = output_dir[0]
    #print((_tiff_images, labels['tickness']
    #        ,missing_slices_list, _output_dir, labels['num_of_slices'].item(), labels['is_curvature'].item(),'glb', device))
    tasks_to_run.append((_tiff_images, labels['tickness'].item()
            ,missing_slices_list, _output_dir, labels['num_of_slices'].item(), labels['is_curvature'].item(),'glb', device))
    
    if len(tasks_to_run) > 10:
        break
multiprocessing.set_start_method('spawn', force=True)
with multiprocessing.Pool( ) as pool: # Use a pool of 4 processes
    pool.starmap(obj_2_pcd.tiff_lilst_2_brain_obj, tqdm(tasks_to_run, total=len(tasks_to_run), desc="_tiff_2_pcd_func"))

print("DONE!")

_tiff_2_pcd_func: 100%|██████████| 11/11 [00:00<00:00, 1591.66it/s]


DONE!
